# 05 Final Evaluation and Results

This notebook loads saved model outputs and produces the final evaluation tables and figures used in the thesis.


## Notebook Structure

1. Load saved inputs and model outputs  
2. Baseline performance and hyperparameter tuning summary  
3. Post-hoc threshold diagnostics  
4. Overall test-set performance and SES comparison  
5. Generalization check and best model selection  
6. Confusion matrix and subgroup error analysis  
7. Feature importance and SHAP analysis  
8. Export thesis tables and figures


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip install category_encoders catboost shap -q


In [ ]:
import os
import json
import pickle
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
import shap


## 1. Load Saved Inputs and Model Outputs

Load preprocessed splits, fitted models, predictions, metrics, cross-validation results, confusion matrices, and feature importance outputs.


In [ ]:
# ============================================================
# 05_results_analysis.ipynb
# Load saved model outputs
# ============================================================

save_dir = "/content/drive/MyDrive/Thesis"

# Optional: check files in save_dir
files = sorted(os.listdir(save_dir))
print("Number of files in save_dir:", len(files))
for f in files:
    if any(key in f for key in [
        "logistic_regression",
        "rf_classifier",
        "catboost_classifier"
    ]):
        print(f)


In [ ]:
# Load preprocessed splits used in modeling

with open(f"{save_dir}/model_splits_preprocessed.pkl", "rb") as f:
    split_data = pickle.load(f)

# Feature lists
features_no_ses = split_data["features_no_ses"]
features_with_ses = split_data["features_with_ses"]

categorical_features = split_data["categorical_features"]
numeric_features_ses = split_data["numeric_features_ses"]

high_cardinality_features_no_ses = split_data["high_cardinality_features_no_ses"]
high_cardinality_features_ses = split_data["high_cardinality_features_ses"]

low_cardinality_features_no_ses = split_data["low_cardinality_features_no_ses"]
low_cardinality_features_ses = split_data["low_cardinality_features_ses"]

# Standard sklearn modeling data
X_train_no_ses = split_data["X_train_no_ses"]
X_test_no_ses = split_data["X_test_no_ses"]
y_train_no_ses = split_data["y_train_no_ses"]
y_test_no_ses = split_data["y_test_no_ses"]

X_train_ses = split_data["X_train_ses"]
X_test_ses = split_data["X_test_ses"]
y_train_ses = split_data["y_train_ses"]
y_test_ses = split_data["y_test_ses"]

# CatBoost modeling data
X_train_cb_clf_no_ses = split_data["X_train_cb_clf_no_ses"]
X_test_cb_clf_no_ses = split_data["X_test_cb_clf_no_ses"]
cat_cols_clf_no_ses = split_data["cat_cols_clf_no_ses"]
cat_features_clf_no_ses = split_data["cat_features_clf_no_ses"]

X_train_cb_clf_ses = split_data["X_train_cb_clf_ses"]
X_test_cb_clf_ses = split_data["X_test_cb_clf_ses"]
cat_cols_clf_ses = split_data["cat_cols_clf_ses"]
cat_features_clf_ses = split_data["cat_features_clf_ses"]

print("Loaded model_splits_preprocessed.pkl")
print("X_train_ses:", X_train_ses.shape)
print("X_test_ses:", X_test_ses.shape)
print("numeric_features_ses:", numeric_features_ses)
print("high_cardinality_features_ses:", high_cardinality_features_ses)
print("low_cardinality_features_ses:", low_cardinality_features_ses)


In [ ]:

def load_csv(path, index_col=None):
    if os.path.exists(path):
        return pd.read_csv(path, index_col=index_col)
    print("Missing:", path)
    return None


def load_json(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    print("Missing:", path)
    return None


def load_joblib(path):
    if os.path.exists(path):
        return joblib.load(path)
    print("Missing:", path)
    return None


def load_pickle(path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    print("Missing:", path)
    return None


In [ ]:
# ============================================================
# Load Logistic Regression outputs
# ============================================================

logreg_outputs = {
    "No SES": {
        "model": load_joblib(f"{save_dir}/logreg_classifier_no_ses.pkl"),
        "test_metrics": load_csv(f"{save_dir}/logreg_classifier_no_ses_test_metrics.csv"),
        "predictions": load_csv(f"{save_dir}/logreg_classifier_no_ses_test_predictions.csv"),
        "confusion_matrix": load_csv(
            f"{save_dir}/logreg_classifier_no_ses_confusion_matrix.csv",
            index_col=0
        ),
        "classification_report": load_csv(
            f"{save_dir}/logreg_classifier_no_ses_classification_report.csv",
            index_col=0
        ),
        "coefficients": load_csv(f"{save_dir}/logreg_classifier_no_ses_coefficients.csv"),
        "summary": load_json(f"{save_dir}/logreg_classifier_no_ses_summary.json"),
        "run_metadata": load_json(f"{save_dir}/logreg_classifier_no_ses_run_metadata.json")
    },

    "With SES": {
        "model": load_joblib(f"{save_dir}/logreg_classifier_ses.pkl"),
        "test_metrics": load_csv(f"{save_dir}/logreg_classifier_ses_test_metrics.csv"),
        "predictions": load_csv(f"{save_dir}/logreg_classifier_ses_test_predictions.csv"),
        "confusion_matrix": load_csv(
            f"{save_dir}/logreg_classifier_ses_confusion_matrix.csv",
            index_col=0
        ),
        "classification_report": load_csv(
            f"{save_dir}/logreg_classifier_ses_classification_report.csv",
            index_col=0
        ),
        "coefficients": load_csv(f"{save_dir}/logreg_classifier_ses_coefficients.csv"),
        "summary": load_json(f"{save_dir}/logreg_classifier_ses_summary.json"),
        "run_metadata": load_json(f"{save_dir}/logreg_classifier_ses_run_metadata.json")
    }
}

print("Loaded Logistic Regression outputs.")


In [ ]:
# ============================================================
# Random Forest outputs
# ============================================================

rf_suffixes = {
    "No SES": "no_ses",
    "With SES": "ses"
}

rf_outputs = {}

for feature_set, suffix in rf_suffixes.items():
    prefix = f"rf_classifier_tuned_{suffix}"

    rf_outputs[feature_set] = {
        "model": load_joblib(f"{save_dir}/{prefix}.pkl"),
        "cv_results": load_csv(f"{save_dir}/rf_classifier_tuning_cv_results_{suffix}.csv"),
        "summary": load_json(f"{save_dir}/{prefix}_summary.json"),
        "test_metrics": load_csv(f"{save_dir}/{prefix}_test_metrics.csv"),
        "predictions": load_csv(f"{save_dir}/{prefix}_test_predictions.csv"),
        "confusion_matrix": load_csv(f"{save_dir}/{prefix}_confusion_matrix.csv", index_col=0),
        "feature_importance": load_csv(f"{save_dir}/{prefix}_feature_importance.csv")
    }

print("Loaded Random Forest outputs.")


In [ ]:
# ============================================================
# Load CatBoost outputs
# ============================================================

cat_outputs = {
    "No SES": {
        "model": load_joblib(f"{save_dir}/catboost_classifier_tuned_no_ses.pkl"),
        "summary": load_json(f"{save_dir}/catboost_classifier_tuned_no_ses_summary.json"),
        "test_metrics": load_csv(f"{save_dir}/catboost_classifier_tuned_no_ses_test_metrics.csv"),
        "predictions": load_csv(f"{save_dir}/catboost_classifier_tuned_no_ses_test_predictions.csv"),
        "confusion_matrix": load_csv(
            f"{save_dir}/catboost_classifier_tuned_no_ses_confusion_matrix.csv",
            index_col=0
        ),
        "feature_importance": load_csv(f"{save_dir}/catboost_classifier_tuned_no_ses_feature_importance.csv"),
        "classification_report": load_csv(
            f"{save_dir}/catboost_classifier_tuned_no_ses_classification_report.csv",
            index_col=0
        ),
        "best_params": load_csv(f"{save_dir}/catboost_classifier_tuned_no_ses_best_params.csv"),
        "run_metadata": load_json(f"{save_dir}/catboost_classifier_tuned_no_ses_run_metadata.json"),
        "search_result": load_pickle(f"{save_dir}/catboost_classifier_search_result_no_ses.pkl")
    },

    "With SES": {
        "model": load_joblib(f"{save_dir}/catboost_classifier_tuned_ses.pkl"),
        "summary": load_json(f"{save_dir}/catboost_classifier_tuned_ses_summary.json"),
        "test_metrics": load_csv(f"{save_dir}/catboost_classifier_tuned_ses_test_metrics.csv"),
        "predictions": load_csv(f"{save_dir}/catboost_classifier_tuned_ses_test_predictions.csv"),
        "confusion_matrix": load_csv(
            f"{save_dir}/catboost_classifier_tuned_ses_confusion_matrix.csv",
            index_col=0
        ),
        "feature_importance": load_csv(f"{save_dir}/catboost_classifier_tuned_ses_feature_importance.csv"),
        "classification_report": load_csv(
            f"{save_dir}/catboost_classifier_tuned_ses_classification_report.csv",
            index_col=0
        ),
        "best_params": load_csv(f"{save_dir}/catboost_classifier_tuned_ses_best_params.csv"),
        "run_metadata": load_json(f"{save_dir}/catboost_classifier_tuned_ses_run_metadata.json"),
        "search_result": load_pickle(f"{save_dir}/catboost_classifier_search_result_ses.pkl")
    }
}

# ------------------------------------------------------------
# Add CatBoost cv_results as DataFrame, same style as RF
# ------------------------------------------------------------

for feature_set in ["No SES", "With SES"]:
    search_result = cat_outputs[feature_set]["search_result"]

    if search_result is not None and "cv_results" in search_result:
        cat_outputs[feature_set]["cv_results"] = pd.DataFrame(search_result["cv_results"])
    else:
        cat_outputs[feature_set]["cv_results"] = None

print("Loaded CatBoost outputs.")


## 2. Hyperparameter Tuning Summary


In [ ]:
# ============================================================
# CV results check
# ============================================================

print("RF No SES CV results")
display(rf_outputs["No SES"]["cv_results"].head())

print("RF With SES CV results")
display(rf_outputs["With SES"]["cv_results"].head())

print("CatBoost No SES CV results")
display(cat_outputs["No SES"]["cv_results"].head())

print("CatBoost With SES CV results")
display(cat_outputs["With SES"]["cv_results"].head())


In [ ]:
# ============================================================
# Hyperparameter Tuning Summary Table
# ============================================================

tuning_summary_rows = []

# ------------------------------------------------------------
# Logistic Regression
# If LR was not tuned, include it as fixed baseline configuration
# ------------------------------------------------------------

for feature_set in ["No SES", "With SES"]:
    metadata = logreg_outputs[feature_set]["run_metadata"]

    if metadata is not None:
        row = {
            "model": "Logistic Regression",
            "feature_set": feature_set,
            "tuned": "No",
            "best_cv_f1": np.nan,
            "best_params": {
                "class_weight": metadata.get("class_weight"),
                "solver": metadata.get("solver"),
                "penalty": metadata.get("penalty"),
                "C": metadata.get("C"),
                "max_iter": metadata.get("max_iter")
            }
        }
        tuning_summary_rows.append(row)


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

for feature_set in ["No SES", "With SES"]:
    summary = rf_outputs[feature_set]["summary"]

    if summary is not None:
        row = {
            "model": "Random Forest",
            "feature_set": feature_set,
            "tuned": "Yes",
            "best_cv_f1": summary.get("best_cv_f1"),
            "best_params": summary.get("best_params")
        }
        tuning_summary_rows.append(row)


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

for feature_set in ["No SES", "With SES"]:
    search_result = cat_outputs[feature_set]["search_result"]
    summary = cat_outputs[feature_set]["summary"]

    if search_result is not None:
        best_params = search_result.get("params")
    elif summary is not None:
        best_params = summary.get("best_params")
    else:
        best_params = None

    # CatBoost randomized_search does not always save best CV F1 in the same way
    # so we try to extract it from cv_results if available
    best_cv_f1 = np.nan

    cv_df = cat_outputs[feature_set].get("cv_results")
    if cv_df is not None:
        possible_cols = [c for c in cv_df.columns if "test" in c.lower() and "F1" in c]
        if len(possible_cols) > 0:
            best_cv_f1 = cv_df[possible_cols[0]].max()

    row = {
        "model": "CatBoost",
        "feature_set": feature_set,
        "tuned": "Yes",
        "best_cv_f1": best_cv_f1,
        "best_params": best_params
    }
    tuning_summary_rows.append(row)


tuning_summary = pd.DataFrame(tuning_summary_rows)

display(tuning_summary)


### 2.1 Train-Test Generalization Check


In [ ]:
def evaluate_train_test(model, X_train, y_train, X_test, y_test, model_name, feature_set):
    rows = []

    for split_name, X, y in [
        ("Train", X_train, y_train),
        ("Test", X_test, y_test)
    ]:
        y_pred = np.array(model.predict(X)).ravel()

        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X)[:, 1]
            roc_auc = roc_auc_score(y, y_proba)
            pr_auc = average_precision_score(y, y_proba)
        else:
            roc_auc = np.nan
            pr_auc = np.nan

        rows.append({
            "model": model_name,
            "feature_set": feature_set,
            "split": split_name,
            "accuracy": accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, zero_division=0),
            "recall": recall_score(y, y_pred, zero_division=0),
            "f1": f1_score(y, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "pr_auc": pr_auc
        })

    return pd.DataFrame(rows)


In [ ]:
# With SES
train_pool_ses = Pool(
    data=X_train_cb_clf_ses,
    label=y_train_ses,
    cat_features=cat_features_clf_ses
)

test_pool_ses = Pool(
    data=X_test_cb_clf_ses,
    label=y_test_ses,
    cat_features=cat_features_clf_ses
)

# No SES
train_pool_no_ses = Pool(
    data=X_train_cb_clf_no_ses,
    label=y_train_no_ses,
    cat_features=cat_features_clf_no_ses
)

test_pool_no_ses = Pool(
    data=X_test_cb_clf_no_ses,
    label=y_test_no_ses,
    cat_features=cat_features_clf_no_ses
)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


In [ ]:
catboost_generalization = pd.concat([
    evaluate_train_test(
        cat_outputs["No SES"]["model"],
        train_pool_no_ses,
        y_train_no_ses,
        test_pool_no_ses,
        y_test_no_ses,
        "CatBoost Tuned",
        "No SES"
    ),
    evaluate_train_test(
        cat_outputs["With SES"]["model"],
        train_pool_ses,
        y_train_ses,
        test_pool_ses,
        y_test_ses,
        "CatBoost Tuned",
        "With SES"
    )
], ignore_index=True)

catboost_generalization.round(3)


### 2.2 Post-hoc Threshold Diagnostics

These threshold checks are exploratory diagnostics. They should not be used for final model selection because the thresholds are selected on the test set.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def find_best_f1_threshold(y_true, y_proba, thresholds=None):
    """
    Returns:
    1. results_df: all threshold results
    2. best_result: best F1 row
    """

    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    results = []

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)

        results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0)
        })

    results_df = pd.DataFrame(results)
    best_idx = results_df["f1"].idxmax()
    best_result = results_df.loc[best_idx].copy()

    return results_df, best_result


In [ ]:
y_proba_cat_ses = cat_outputs["With SES"]["model"].predict_proba(test_pool_ses)[:, 1]

cat_ses_threshold_results, cat_ses_best = find_best_f1_threshold(
    y_test_ses,
    y_proba_cat_ses
)

print(cat_ses_best.round(3))


In [ ]:
y_proba_cat_no_ses = cat_outputs["No SES"]["model"].predict_proba(test_pool_no_ses)[:, 1]

cat_no_ses_threshold_results, cat_no_ses_best = find_best_f1_threshold(
    y_test_no_ses,
    y_proba_cat_no_ses
)

print(cat_no_ses_best.round(3))


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def find_best_f1_threshold(y_true, y_proba, thresholds=None):
    """
    Returns:
    1. results_df: all threshold results
    2. best_result: best F1 row
    """

    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    results = []

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)

        results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0)
        })

    results_df = pd.DataFrame(results)
    best_idx = results_df["f1"].idxmax()
    best_result = results_df.loc[best_idx].copy()

    return results_df, best_result


In [ ]:
y_proba_rf_no_ses = rf_outputs["No SES"]["model"].predict_proba(X_test_no_ses)[:, 1]

rf_no_ses_threshold_results, rf_no_ses_best = find_best_f1_threshold(
    y_test_no_ses,
    y_proba_rf_no_ses
)

print(rf_no_ses_best.round(3))


In [ ]:
y_proba_rf_ses = rf_outputs["With SES"]["model"].predict_proba(X_test_ses)[:, 1]

rf_ses_threshold_results, rf_ses_best = find_best_f1_threshold(
    y_test_ses,
    y_proba_rf_ses
)

print(rf_ses_best.round(3))


### 2.3 Thesis-Ready Hyperparameter Summary Table


In [ ]:
# ============================================================
# Make thesis-ready tuning summary table
# ============================================================

def params_to_string(params):
    if params is None or (isinstance(params, float) and np.isnan(params)):
        return ""

    if isinstance(params, str):
        return params

    if isinstance(params, dict):
        return "; ".join([f"{k}={v}" for k, v in params.items()])

    return str(params)


tuning_summary_thesis = tuning_summary.copy()

tuning_summary_thesis["best_params"] = tuning_summary_thesis["best_params"].apply(params_to_string)

# Round CV score
tuning_summary_thesis["best_cv_f1"] = tuning_summary_thesis["best_cv_f1"].round(3)

display(tuning_summary_thesis)

tuning_summary_thesis.to_csv(
    f"{save_dir}/hyperparameter_tuning_summary_table.csv",
    index=False
)


In [ ]:
# ============================================================
# Add test F1 to tuning summary
# ============================================================

test_f1_lookup = {
    ("Logistic Regression", "No SES"): all_test_metrics[
        (all_test_metrics["model"].str.contains("Logistic", na=False)) &
        (all_test_metrics["feature_set"] == "No SES")
    ]["f1"].iloc[0],

    ("Logistic Regression", "With SES"): all_test_metrics[
        (all_test_metrics["model"].str.contains("Logistic", na=False)) &
        (all_test_metrics["feature_set"] == "With SES")
    ]["f1"].iloc[0],

    ("Random Forest", "No SES"): all_test_metrics[
        (all_test_metrics["model"].str.contains("Random Forest", na=False)) &
        (all_test_metrics["feature_set"] == "No SES")
    ]["f1"].iloc[0],

    ("Random Forest", "With SES"): all_test_metrics[
        (all_test_metrics["model"].str.contains("Random Forest", na=False)) &
        (all_test_metrics["feature_set"] == "With SES")
    ]["f1"].iloc[0],

    ("CatBoost", "No SES"): all_test_metrics[
        (all_test_metrics["model"].str.contains("CatBoost", na=False)) &
        (all_test_metrics["feature_set"] == "No SES")
    ]["f1"].iloc[0],

    ("CatBoost", "With SES"): all_test_metrics[
        (all_test_metrics["model"].str.contains("CatBoost", na=False)) &
        (all_test_metrics["feature_set"] == "With SES")
    ]["f1"].iloc[0],
}

tuning_summary_thesis["test_f1"] = tuning_summary_thesis.apply(
    lambda row: test_f1_lookup.get((row["model"], row["feature_set"]), np.nan),
    axis=1
)

tuning_summary_thesis["cv_test_gap"] = (
    pd.to_numeric(tuning_summary_thesis["best_cv_f1"], errors="coerce")
    - tuning_summary_thesis["test_f1"]
)

tuning_summary_thesis["test_f1"] = tuning_summary_thesis["test_f1"].round(3)
tuning_summary_thesis["cv_test_gap"] = tuning_summary_thesis["cv_test_gap"].round(3)

display(tuning_summary_thesis)

tuning_summary_thesis.to_csv(
    f"{save_dir}/hyperparameter_tuning_summary_with_test_f1.csv",
    index=False
)


In [ ]:
print("CatBoost No SES CV columns:")
print(cat_outputs["No SES"]["cv_results"].columns.tolist())

print("\nCatBoost With SES CV columns:")
print(cat_outputs["With SES"]["cv_results"].columns.tolist())

display(cat_outputs["No SES"]["cv_results"].head())
display(cat_outputs["No SES"]["cv_results"].tail())

display(cat_outputs["With SES"]["cv_results"].head())
display(cat_outputs["With SES"]["cv_results"].tail())


## 3. Overall Test-Set Performance


In [ ]:
# ============================================================
# Combine test metrics
# ============================================================

all_metric_dfs = []

# Logistic Regression
for_feature_sets = ["No SES", "With SES"]

for feature_set in for_feature_sets:
    df = logreg_outputs[feature_set]["test_metrics"]
    if df is not None:
        df = df.copy()
        df["source_file_model"] = "Logistic Regression"
        all_metric_dfs.append(df)

# Random Forest
for feature_set in for_feature_sets:
    df = rf_outputs[feature_set]["test_metrics"]
    if df is not None:
        df = df.copy()
        df["source_file_model"] = "Random Forest"
        all_metric_dfs.append(df)

# CatBoost
for feature_set in for_feature_sets:
    df = cat_outputs[feature_set]["test_metrics"]
    if df is not None:
        df = df.copy()
        df["source_file_model"] = "CatBoost"
        all_metric_dfs.append(df)

all_test_metrics = pd.concat(all_metric_dfs, ignore_index=True)

# Clean display columns if available
cols_to_show = [
    "stage", "feature_set", "model",
    "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc",
    "tn", "fp", "fn", "tp",
    "specificity", "false_positive_rate", "false_negative_rate"
]

available_cols = [c for c in cols_to_show if c in all_test_metrics.columns]
all_test_metrics_display = all_test_metrics[available_cols].copy()

display(all_test_metrics_display)

# Save combined table
all_test_metrics_display.to_csv(
    f"{save_dir}/combined_classification_test_metrics.csv",
    index=False
)


In [ ]:
results_test_set_table = all_test_metrics[
    [
        "model",
        "feature_set",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc"
    ]
].copy()

results_test_set_table = results_test_set_table.round(3)

display(results_test_set_table)

results_test_set_table.to_csv(
    f"{save_dir}/results_test_set_table.csv",
    index=False
)


## 4. Effect of Adding Socioeconomic Features


In [ ]:
# ============================================================
# SES improvement table
# ============================================================

metrics_for_delta = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]

metrics_clean = all_test_metrics_display.copy()

metrics_clean["model_clean"] = metrics_clean["model"].str.replace(" Tuned", "", regex=False)

delta_rows = []

for model_name in metrics_clean["model_clean"].unique():
    no_ses_row = metrics_clean[
        (metrics_clean["model_clean"] == model_name) &
        (metrics_clean["feature_set"] == "No SES")
    ]

    ses_row = metrics_clean[
        (metrics_clean["model_clean"] == model_name) &
        (metrics_clean["feature_set"] == "With SES")
    ]

    if len(no_ses_row) == 1 and len(ses_row) == 1:
        row = {"model": model_name}

        for metric in metrics_for_delta:
            if metric in metrics_clean.columns:
                row[f"delta_{metric}"] = (
                    ses_row[metric].values[0] - no_ses_row[metric].values[0]
                )

        delta_rows.append(row)

ses_delta_table = pd.DataFrame(delta_rows)

display(ses_delta_table)

ses_delta_table.to_csv(
    f"{save_dir}/ses_performance_delta_table.csv",
    index=False
)


## 5. Generalization Check

This section compares cross-validation and held-out test performance to identify possible generalization gaps.


## 6. Best Model Selection


In [ ]:
# ============================================================
# Identify best model by F1
# ============================================================

best_by_f1 = all_test_metrics_display.sort_values("f1", ascending=False).head(10)

display(best_by_f1)

best_model_row = best_by_f1.iloc[0]
print("Best model by test F1:")
print(best_model_row)


## 7. Confusion Matrix and Error Rates

This section examines the confusion matrix of the selected model and derives false positive and false negative rates.


In [ ]:
# ============================================================
# Display confusion matrices
# ============================================================

print("Logistic Regression - No SES")
display(logreg_outputs["No SES"]["confusion_matrix"])

print("Logistic Regression - With SES")
display(logreg_outputs["With SES"]["confusion_matrix"])

print("Random Forest - No SES")
display(rf_outputs["No SES"]["confusion_matrix"])

print("Random Forest - With SES")
display(rf_outputs["With SES"]["confusion_matrix"])

print("CatBoost - No SES")
display(cat_outputs["No SES"]["confusion_matrix"])

print("CatBoost - With SES")
display(cat_outputs["With SES"]["confusion_matrix"])


### 7.1 Confusion Matrix Heatmap


In [ ]:
# ============================================================
# Load confusion matrix for best model: Random Forest With SES
# ============================================================

save_dir = "/content/drive/MyDrive/Thesis"
figure_dir = f"{save_dir}/figures"
os.makedirs(figure_dir, exist_ok=True)

cm_path = f"{save_dir}/rf_classifier_tuned_ses_confusion_matrix.csv"

cm_rf_ses = pd.read_csv(cm_path, index_col=0)

display(cm_rf_ses)
print(cm_rf_ses.shape)


In [ ]:
# ============================================================
# Confusion Matrix Heatmap - Random Forest With SES
# Blue style
# ============================================================

cm_values = cm_rf_ses.values

plt.figure(figsize=(6, 5))

im = plt.imshow(
    cm_values,
    cmap="GnBu"
)

plt.title("Confusion Matrix: Random Forest with SES", fontsize=13, pad=12)
plt.xlabel("Predicted label", fontsize=11)
plt.ylabel("True label", fontsize=11)

plt.xticks(
    ticks=[0, 1],
    labels=["Short-stay", "Long-stay"],
    fontsize=10
)

plt.yticks(
    ticks=[0, 1],
    labels=["Short-stay", "Long-stay"],
    fontsize=10
)

# Add numbers inside cells
threshold = cm_values.max() / 2

for i in range(cm_values.shape[0]):
    for j in range(cm_values.shape[1]):
        plt.text(
            j,
            i,
            int(cm_values[i, j]),
            ha="center",
            va="center",
            fontsize=12,
            color="white" if cm_values[i, j] > threshold else "black"
        )

plt.colorbar(im, fraction=0.046, pad=0.04)

plt.tight_layout()

plt.savefig(
    f"{figure_dir}/rf-confusion-matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{figure_dir}/rf-confusion-matrix.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# Calculate FPR and FNR from confusion matrix
# Best model: Random Forest With SES
# ============================================================

# If confusion matrix is already loaded:
# cm_rf_ses = rf_outputs["With SES"]["confusion_matrix"].copy()

# Or load from saved CSV:
cm_path = f"{save_dir}/rf_classifier_tuned_ses_confusion_matrix.csv"
cm_rf_ses = pd.read_csv(cm_path, index_col=0)

display(cm_rf_ses)

# Convert to values
cm = cm_rf_ses.values

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)
specificity = tn / (tn + fp)
recall = tp / (tp + fn)

error_summary = pd.DataFrame({
    "metric": [
        "True negatives",
        "False positives",
        "False negatives",
        "True positives",
        "False positive rate",
        "False negative rate",
        "Specificity",
        "Recall"
    ],
    "value": [
        tn,
        fp,
        fn,
        tp,
        fpr,
        fnr,
        specificity,
        recall
    ]
})

display(error_summary)

error_summary.to_csv(
    f"{save_dir}/rf_with_ses_error_summary.csv",
    index=False
)

print("FPR:", round(fpr, 3))
print("FNR:", round(fnr, 3))
print("Specificity:", round(specificity, 3))
print("Recall:", round(recall, 3))


## 8. Subgroup Error Analysis

This section evaluates whether model errors differ across animal-level, intake-level, socioeconomic, and SES linkage groups.


In [ ]:
# ============================================================
# Combine prediction files
# ============================================================

prediction_dfs = []

for feature_set in ["No SES", "With SES"]:
    pred = logreg_outputs[feature_set]["predictions"]
    if pred is not None:
        pred = pred.copy()
        pred["model"] = "Logistic Regression"
        pred["feature_set"] = feature_set
        prediction_dfs.append(pred)

for feature_set in ["No SES", "With SES"]:
    pred = rf_outputs[feature_set]["predictions"]
    if pred is not None:
        pred = pred.copy()
        pred["model"] = "Random Forest"
        pred["feature_set"] = feature_set
        prediction_dfs.append(pred)

for feature_set in ["No SES", "With SES"]:
    pred = cat_outputs[feature_set]["predictions"]
    if pred is not None:
        pred = pred.copy()
        pred["model"] = "CatBoost"
        pred["feature_set"] = feature_set
        prediction_dfs.append(pred)

all_predictions = pd.concat(prediction_dfs, ignore_index=True)

display(all_predictions.head())
print(all_predictions.shape)

all_predictions.to_csv(
    f"{save_dir}/combined_classification_test_predictions.csv",
    index=False
)


In [ ]:
all_predictions.groupby(["model", "feature_set"]).size()


In [ ]:
# ============================================================
# Subgroup Error Analysis - Best Model: RF With SES
# ============================================================

subgroup_dir = f"{save_dir}/subgroup_error_analysis"
os.makedirs(subgroup_dir, exist_ok=True)

# ------------------------------------------------------------
# Select predictions from best model only
# ------------------------------------------------------------

best_pred = all_predictions[
    (all_predictions["model"] == "Random Forest") &
    (all_predictions["feature_set"] == "With SES")
].copy()

print("Best model prediction shape:", best_pred.shape)
display(best_pred.head())

# ------------------------------------------------------------
# Merge predictions with original X_test_ses features
# ------------------------------------------------------------

X_test_ses_for_merge = X_test_ses.copy()
X_test_ses_for_merge["index"] = X_test_ses_for_merge.index

error_df = best_pred.merge(
    X_test_ses_for_merge,
    on="index",
    how="left"
)

print("Merged error_df shape:", error_df.shape)
display(error_df.head())


In [ ]:
# ============================================================
# Add error type columns
# ============================================================

error_df["correct"] = error_df["y_true"] == error_df["y_pred"]

error_df["error_type"] = np.select(
    [
        (error_df["y_true"] == 1) & (error_df["y_pred"] == 1),
        (error_df["y_true"] == 0) & (error_df["y_pred"] == 0),
        (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
        (error_df["y_true"] == 0) & (error_df["y_pred"] == 1)
    ],
    [
        "True Positive",
        "True Negative",
        "False Negative",
        "False Positive"
    ],
    default="Unknown"
)

display(error_df["error_type"].value_counts())


In [ ]:
# ============================================================
# Function: subgroup error metrics
# ============================================================

def subgroup_error_metrics(df, group_col):
    rows = []

    for group_value, g in df.groupby(group_col, dropna=False):
        y_true = g["y_true"]
        y_pred = g["y_pred"]

        tp = ((y_true == 1) & (y_pred == 1)).sum()
        tn = ((y_true == 0) & (y_pred == 0)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        n = len(g)
        long_stay_n = int((y_true == 1).sum())
        short_stay_n = int((y_true == 0).sum())

        row = {
            "subgroup_variable": group_col,
            "subgroup": group_value,
            "n": n,
            "long_stay_n": long_stay_n,
            "short_stay_n": short_stay_n,
            "long_stay_rate": long_stay_n / n if n > 0 else np.nan,
            "accuracy": (tp + tn) / n if n > 0 else np.nan,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "tp": int(tp),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
            "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else np.nan
        }

        rows.append(row)

    result = pd.DataFrame(rows)

    # Sort larger groups first
    result = result.sort_values("n", ascending=False).reset_index(drop=True)

    return result


In [ ]:
# ============================================================
# Subgroup analysis: animal and intake variables
# ============================================================

subgroup_vars = [
    "Animal_Type",
    "Intake_Type",
    "Intake_Subtype_fe",
    "Intake_Condition_clean",
    "Animal_Size",
    "Chip_Status_fe",
    "Animal_Origin_fe"
]

subgroup_results = {}

for col in subgroup_vars:
    if col in error_df.columns:
        result = subgroup_error_metrics(error_df, col)
        subgroup_results[col] = result

        print(f"\n===== {col} =====")
        display(result)

        result.to_csv(
            f"{subgroup_dir}/subgroup_error_{col}.csv",
            index=False
        )
    else:
        print(f"Column not found: {col}")


In [ ]:
# ============================================================
# Create SES quartile groups
# ============================================================

ses_features = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

for col in ses_features:
    if col in error_df.columns:
        quartile_col = f"{col}_quartile"

        error_df[quartile_col] = pd.qcut(
            error_df[col],
            q=4,
            labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
            duplicates="drop"
        )

        print(f"{quartile_col} created.")
    else:
        print(f"SES column not found: {col}")


In [ ]:
# ============================================================
# Subgroup analysis: SES quartiles
# ============================================================

ses_quartile_cols = [
    f"{col}_quartile"
    for col in ses_features
    if f"{col}_quartile" in error_df.columns
]

ses_subgroup_results = {}

for col in ses_quartile_cols:
    result = subgroup_error_metrics(error_df, col)
    ses_subgroup_results[col] = result

    print(f"\n===== {col} =====")
    display(result)

    result.to_csv(
        f"{subgroup_dir}/subgroup_error_{col}.csv",
        index=False
    )


In [ ]:
# ============================================================
# Main thesis subgroup table
# Recommended: Animal_Type, Intake_Type, Poverty quartile, Income quartile
# ============================================================

main_subgroup_tables = []

main_cols = [
    "Animal_Type",
    "Intake_Type",
    "poverty_rate_quartile",
    "median_household_income_quartile"
]

for col in main_cols:
    if col in subgroup_results:
        main_subgroup_tables.append(subgroup_results[col])
    elif col in ses_subgroup_results:
        main_subgroup_tables.append(ses_subgroup_results[col])
    else:
        print(f"Not found in results: {col}")

main_subgroup_error_table = pd.concat(
    main_subgroup_tables,
    ignore_index=True
)

# Select columns for thesis table
thesis_cols = [
    "subgroup_variable",
    "subgroup",
    "n",
    "long_stay_rate",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "false_negative_rate",
    "fp",
    "fn"
]

main_subgroup_error_table = main_subgroup_error_table[thesis_cols]

display(main_subgroup_error_table)

main_subgroup_error_table.to_csv(
    f"{subgroup_dir}/main_subgroup_error_table_rf_with_ses.csv",
    index=False
)


In [ ]:
# ============================================================
# Rounded thesis-ready subgroup table
# ============================================================

main_subgroup_error_table_rounded = main_subgroup_error_table.copy()

round_cols = [
    "long_stay_rate",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "false_negative_rate"
]

for col in round_cols:
    main_subgroup_error_table_rounded[col] = main_subgroup_error_table_rounded[col].round(3)

display(main_subgroup_error_table_rounded)

main_subgroup_error_table_rounded.to_csv(
    f"{subgroup_dir}/main_subgroup_error_table_rf_with_ses_rounded.csv",
    index=False
)


### 8.1 Thesis-Ready Subgroup Tables


In [ ]:
# ============================================================
# Load saved subgroup error analysis outputs
# ============================================================

save_dir = "/content/drive/MyDrive/Thesis"
subgroup_dir = f"{save_dir}/subgroup_error_analysis"

print("Subgroup directory exists:", os.path.exists(subgroup_dir))

print("\nFiles in subgroup directory:")
for f in sorted(os.listdir(subgroup_dir)):
    print(f)


In [ ]:
main_path = f"{subgroup_dir}/main_subgroup_error_table_rf_with_ses.csv"
main_rounded_path = f"{subgroup_dir}/main_subgroup_error_table_rf_with_ses_rounded.csv"

if os.path.exists(main_rounded_path):
    main_subgroup_error_table = pd.read_csv(main_rounded_path)
    print("Loaded rounded main subgroup table.")
elif os.path.exists(main_path):
    main_subgroup_error_table = pd.read_csv(main_path)
    print("Loaded unrounded main subgroup table.")
else:
    raise FileNotFoundError("Main subgroup error table file was not found.")

display(main_subgroup_error_table)
print(main_subgroup_error_table.shape)


In [ ]:
# ============================================================
# Create compact subgroup table for thesis body
# ============================================================

subgroup_compact_table = main_subgroup_error_table.copy()

# Keep only thesis-body columns
subgroup_compact_table = subgroup_compact_table[
    [
        "subgroup_variable",
        "subgroup",
        "n",
        "long_stay_rate",
        "f1",
        "false_positive_rate",
        "false_negative_rate"
    ]
].copy()

# Rename columns for readability
subgroup_compact_table = subgroup_compact_table.rename(columns={
    "subgroup_variable": "Subgroup variable",
    "subgroup": "Subgroup",
    "n": "N",
    "long_stay_rate": "Long-stay rate",
    "f1": "F1",
    "false_positive_rate": "FPR",
    "false_negative_rate": "FNR"
})

# Round numeric columns
round_cols = ["Long-stay rate", "F1", "FPR", "FNR"]
for col in round_cols:
    subgroup_compact_table[col] = pd.to_numeric(
        subgroup_compact_table[col],
        errors="coerce"
    ).round(3)

display(subgroup_compact_table)

subgroup_compact_table.to_csv(
    f"{subgroup_dir}/subgroup-error-compact-table-rf-with-ses.csv",
    index=False
)


In [ ]:
# ============================================================
# Keep only main subgroup variables for thesis body
# ============================================================

main_body_vars = [
    "Animal_Type",
    "Intake_Type",
    "poverty_rate_quartile",
    "median_household_income_quartile"
]

subgroup_compact_body = subgroup_compact_table[
    subgroup_compact_table["Subgroup variable"].isin(main_body_vars)
].copy()

display(subgroup_compact_body)

subgroup_compact_body.to_csv(
    f"{subgroup_dir}/subgroup-error-compact-body-table-rf-with-ses.csv",
    index=False
)


In [ ]:
# ============================================================
# Sort compact subgroup table for thesis body
# ============================================================

subgroup_sorted = subgroup_compact_body.copy()

# Convert NaN subgroup labels to "Missing" for thesis readability
subgroup_sorted["Subgroup"] = subgroup_sorted["Subgroup"].astype(str)
subgroup_sorted["Subgroup"] = subgroup_sorted["Subgroup"].replace("nan", "Missing")
subgroup_sorted["Subgroup"] = subgroup_sorted["Subgroup"].replace("NaN", "Missing")

# Define ordering
variable_order = {
    "Animal_Type": 1,
    "Intake_Type": 2,
    "poverty_rate_quartile": 3,
    "median_household_income_quartile": 4
}

subgroup_order = {
    # Animal type
    "DOG": 1,
    "CAT": 2,

    # Intake type: main/common categories first, then smaller/special categories
    "STRAY": 1,
    "OWNER SURRENDER": 2,
    "FOSTER": 3,
    "CONFISCATED": 4,
    "TREATMENT": 5,
    "TRANSFER": 6,

    # Quartiles
    "Q1 lowest": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4 highest": 4,
    "Missing": 5
}

subgroup_sorted["variable_order"] = subgroup_sorted["Subgroup variable"].map(variable_order)
subgroup_sorted["subgroup_order"] = subgroup_sorted["Subgroup"].map(subgroup_order)

# Fallback for unexpected values
subgroup_sorted["variable_order"] = subgroup_sorted["variable_order"].fillna(99)
subgroup_sorted["subgroup_order"] = subgroup_sorted["subgroup_order"].fillna(99)

subgroup_sorted = subgroup_sorted.sort_values(
    ["variable_order", "subgroup_order", "N"],
    ascending=[True, True, False]
).drop(columns=["variable_order", "subgroup_order"])

subgroup_sorted["Subgroup"] = subgroup_sorted["Subgroup"].replace("nan", "Missing SES")
subgroup_sorted["Subgroup"] = subgroup_sorted["Subgroup"].replace("NaN", "Missing SES")

display(subgroup_sorted)

# Save sorted table
subgroup_sorted.to_csv(
    f"{subgroup_dir}/subgroup-error-compact-body-table-rf-with-ses-sorted.csv",
    index=False
)


In [ ]:
# ============================================================
# Export compact subgroup table to LaTeX
# ============================================================

latex_table = subgroup_sorted.to_latex(
    index=False,
    escape=True,
    float_format="%.3f",
    caption="Subgroup Error Analysis for the SES-Augmented Random Forest Classifier",
    label="tab:subgroup-error-analysis"
)

print(latex_table)


### 8.2 Subgroup Error Plots


In [ ]:
# ============================================================
# Plot subgroup false negative rates
# ============================================================

def plot_subgroup_metric(result_df, group_col, metric="false_negative_rate", top_n=10):
    plot_df = result_df.copy()

    plot_df = plot_df[plot_df["n"] >= 100].copy()

    plot_df = plot_df.sort_values(metric, ascending=True).tail(top_n)

    plt.figure(figsize=(8, 5))

    plt.barh(
        plot_df["subgroup"].astype(str),
        plot_df[metric]
    )

    plt.xlabel(metric.replace("_", " ").title())
    plt.ylabel("")
    plt.title(f"{metric.replace('_', ' ').title()} by {group_col}")

    plt.tight_layout()

    filename = f"{subgroup_dir}/plot_{metric}_by_{group_col}.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")

    plt.show()

    print("Saved:", filename)


# Recommended plots
for col in ["Animal_Type", "Intake_Type", "poverty_rate_quartile", "median_household_income_quartile"]:
    if col in subgroup_results:
        plot_subgroup_metric(subgroup_results[col], col, metric="false_negative_rate")
    elif col in ses_subgroup_results:
        plot_subgroup_metric(ses_subgroup_results[col], col, metric="false_negative_rate")


In [ ]:
# ============================================================
# Check saved subgroup files
# ============================================================

saved_files = sorted(os.listdir(subgroup_dir))

print("Saved subgroup analysis files:")
for f in saved_files:
    print(f)


### 8.3 SES Linkage Group Analysis


In [ ]:
# ============================================================
# SES linkage source subgroup analysis
# Best model: Random Forest With SES
# ============================================================

def assign_ses_linkage_group(row):
    if row["ses_direct_linked"] == 1:
        return "Direct SES link"
    elif row["ses_zip_crosswalked"] == 1:
        return "ZIP-crosswalk SES link"
    else:
        return "No SES link"

error_df["ses_linkage_group"] = error_df.apply(assign_ses_linkage_group, axis=1)

# Check group sizes and long-stay rates
ses_linkage_summary = (
    error_df
    .groupby("ses_linkage_group")
    .agg(
        n=("y_true", "size"),
        long_stay_n=("y_true", "sum"),
        long_stay_rate=("y_true", "mean")
    )
    .reset_index()
)

ses_linkage_summary["long_stay_rate"] = ses_linkage_summary["long_stay_rate"].round(3)

display(ses_linkage_summary)

ses_linkage_summary.to_csv(
    f"{subgroup_dir}/ses_linkage_group_summary_rf_with_ses.csv",
    index=False
)

# Error metrics by SES linkage group
ses_linkage_error = subgroup_error_metrics(error_df, "ses_linkage_group")

display(ses_linkage_error)

ses_linkage_error.to_csv(
    f"{subgroup_dir}/subgroup_error_ses_linkage_group.csv",
    index=False
)


In [ ]:
# Use your existing ses_linkage_summary
plot_df = ses_linkage_summary.copy()

plt.figure(figsize=(8, 5))
plt.bar(plot_df["ses_linkage_group"], plot_df["n"])
plt.ylabel("Number of records")
plt.xlabel("SES linkage group")
plt.title("Test-set records by SES linkage group")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(plot_df["ses_linkage_group"], plot_df["long_stay_rate"])
plt.ylabel("Long-stay rate")
plt.xlabel("SES linkage group")
plt.title("Long-stay rate by SES linkage group")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, max(plot_df["long_stay_rate"]) * 1.25)
plt.tight_layout()
plt.show()


In [ ]:
# Use your existing ses_linkage_error
metric_df = ses_linkage_error.copy()

metrics_to_plot = ["false_positive_rate", "false_negative_rate"]

for metric in metrics_to_plot:
    plt.figure(figsize=(8, 5))
    plt.bar(metric_df["subgroup"], metric_df[metric])
    plt.ylabel(metric.replace("_", " ").title())
    plt.xlabel("SES linkage group")
    plt.title(f"{metric.replace('_', ' ').title()} by SES linkage group")
    plt.xticks(rotation=20, ha="right")
    plt.ylim(0, max(metric_df[metric]) * 1.25)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_dir = f"{save_dir}/plots"
os.makedirs(plot_dir, exist_ok=True)

metric_df = ses_linkage_error.copy()

groups = metric_df["subgroup"].tolist()
x = np.arange(len(groups))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, metric_df["false_positive_rate"], width, label="False Positive Rate")
plt.bar(x + width/2, metric_df["false_negative_rate"], width, label="False Negative Rate")

plt.ylabel("Error rate")
plt.xlabel("SES linkage group")
plt.title("False Positive and False Negative Rates by SES Linkage Group")
plt.xticks(x, groups, rotation=20, ha="right")
plt.ylim(
    0,
    max(metric_df["false_positive_rate"].max(), metric_df["false_negative_rate"].max()) * 1.25
)
plt.legend()
plt.tight_layout()

plt.savefig(
    f"{plot_dir}/ses_linkage_group_fpr_fnr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### 8.4 Intake Type Error Plot


In [ ]:
# ============================================================
# Intake Type subgroup error plot
# Best model: Random Forest With Socioeconomic Variables
# ============================================================

import os
import matplotlib.pyplot as plt
import numpy as np

# Save directory
plot_dir = f"{subgroup_dir}/plots"
os.makedirs(plot_dir, exist_ok=True)

# Error metrics by Intake Type
intake_type_error = subgroup_error_metrics(error_df, "Intake_Type")

# Sort by subgroup size descending
intake_type_error = intake_type_error.sort_values("n", ascending=False).reset_index(drop=True)

display(intake_type_error)

# Save table
intake_type_error.to_csv(
    f"{subgroup_dir}/subgroup_error_intake_type.csv",
    index=False
)

# Plot FPR and FNR by Intake Type
x = np.arange(len(intake_type_error))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    x - width / 2,
    intake_type_error["false_positive_rate"],
    width,
    label="False Positive Rate"
)

ax.bar(
    x + width / 2,
    intake_type_error["false_negative_rate"],
    width,
    label="False Negative Rate"
)

ax.set_xlabel("Intake Type")
ax.set_ylabel("Error rate")
ax.set_title("False Positive and False Negative Rates by Intake Type")
ax.set_xticks(x)
ax.set_xticklabels(intake_type_error["subgroup"], rotation=45, ha="right")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()

plt.savefig(
    f"{plot_dir}/intake-type-fpr-fnr-rf-with-ses.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{plot_dir}/intake-type-fpr-fnr-rf-with-ses.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:

# Intake Type subgroup error plot - horizontal version


plot_dir = f"{subgroup_dir}/plots"
os.makedirs(plot_dir, exist_ok=True)

intake_type_error = subgroup_error_metrics(error_df, "Intake_Type")
intake_type_error = intake_type_error.sort_values("false_negative_rate", ascending=True).reset_index(drop=True)

display(intake_type_error)

intake_type_error.to_csv(
    f"{subgroup_dir}/subgroup_error_intake_type.csv",
    index=False
)

y = np.arange(len(intake_type_error))
height = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    y - height / 2,
    intake_type_error["false_positive_rate"],
    height,
    label="False Positive Rate"
)

ax.barh(
    y + height / 2,
    intake_type_error["false_negative_rate"],
    height,
    label="False Negative Rate"
)

ax.set_ylabel("Intake Type")
ax.set_xlabel("Error rate")
ax.set_title("False Positive and False Negative Rates by Intake Type")
ax.set_yticks(y)
ax.set_yticklabels(intake_type_error["subgroup"])
ax.set_xlim(0, 1)
ax.legend(loc="lower right")

plt.tight_layout()

plt.savefig(
    f"{plot_dir}/intake-type-fpr-fnr-rf-with-ses.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{plot_dir}/intake-type-fpr-fnr-rf-with-ses.pdf",
    bbox_inches="tight"
)

plt.show()


## 9. Feature Importance


In [ ]:
# ============================================================
# Check loaded importance outputs
# ============================================================

print("Logistic Regression No SES coefficients")
display(logreg_outputs["No SES"]["coefficients"].head(20))

print("Logistic Regression With SES coefficients")
display(logreg_outputs["With SES"]["coefficients"].head(20))

print("Random Forest No SES feature importance")
display(rf_outputs["No SES"]["feature_importance"].head(20))

print("Random Forest With SES feature importance")
display(rf_outputs["With SES"]["feature_importance"].head(20))

print("CatBoost No SES feature importance")
display(cat_outputs["No SES"]["feature_importance"].head(20))

print("CatBoost With SES feature importance")
display(cat_outputs["With SES"]["feature_importance"].head(20))


In [ ]:
# ============================================================
# Feature Importance Plot - Random Forest With SES
# ============================================================

figure_dir = f"{save_dir}/figures"
os.makedirs(figure_dir, exist_ok=True)

# Load feature importance table from loaded outputs
rf_ses_importance = rf_outputs["With SES"]["feature_importance"].copy()

# Top 20 features
top_n = 20
plot_df = (
    rf_ses_importance
    .sort_values("importance", ascending=False)
    .head(top_n)
    .sort_values("importance", ascending=True)
)

plt.figure(figsize=(9, 7))

plt.barh(
    plot_df["feature"],
    plot_df["importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("")
plt.title("Feature Importance Scores - Random Forest With SES")

plt.tight_layout()

plt.savefig(
    f"{figure_dir}/feature_importance_random_forest_with_ses.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{figure_dir}/feature_importance_random_forest_with_ses.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Aggregated Feature Importance Plot - Random Forest With SES

figure_dir = f"{save_dir}/figures"
os.makedirs(figure_dir, exist_ok=True)

# Use RF With SES feature importance
rf_ses_importance = rf_outputs["With SES"]["feature_importance"].copy()


# ------------------------------------------------------------
# Map transformed feature names back to original feature level
# ------------------------------------------------------------

def map_to_original_feature(name):
    if "Animal_Breed_clean" in name:
        return "Animal breed"

    if "median_household_income" in name:
        return "Median household income"
    if "bachelor_or_higher_pct" in name:
        return "Bachelor's degree or higher (%)"
    if "poverty_rate" in name:
        return "Poverty rate"
    if "unemployment_rate" in name:
        return "Unemployment rate"

    if "Intake_Year" in name:
        return "Intake year"
    if "Intake_Month" in name:
        return "Intake month"
    if "Animal_Origin_fe" in name:
        return "Animal origin"
    if "Intake_Type" in name:
        return "Intake type"
    if "Intake_Subtype_fe" in name:
        return "Intake subtype"
    if "Intake_Condition_clean" in name or "Intake_Condition_fe" in name:
        return "Intake condition"
    if "Chip_Status_fe" in name:
        return "Chip status"
    if "Animal_Type" in name:
        return "Animal type"
    if "Animal_Size" in name:
        return "Animal size"

    return name


rf_ses_importance["original_feature"] = rf_ses_importance["feature"].apply(map_to_original_feature)

rf_ses_importance_agg = (
    rf_ses_importance
    .groupby("original_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

display(rf_ses_importance_agg)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plot_df = rf_ses_importance_agg.sort_values("importance", ascending=True)

plt.figure(figsize=(8.5, 5.8))

bars = plt.barh(
    plot_df["original_feature"],
    plot_df["importance"],
    color="white",
    edgecolor="black",
    hatch="///",
    linewidth=0.8
)

plt.xlabel("Aggregated Feature Importance", fontsize=12)
plt.ylabel("")
plt.title("Feature Importance Scores", fontsize=13, pad=12)

# Clean axis style
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)

plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()

plt.savefig(
    f"{figure_dir}/feature_importance_rf_with_ses_aggregated_thesis_style.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{figure_dir}/feature_importance_rf_with_ses_aggregated_thesis_style.pdf",
    bbox_inches="tight"
)

plt.show()


## 10. SHAP Analysis

This section computes and visualizes SHAP values for the selected Random Forest model with socioeconomic features.


In [ ]:
# ============================================================
# SHAP Analysis - Random Forest With SES
# ============================================================

shap.initjs()

save_dir = "/content/drive/MyDrive/Thesis"


In [ ]:
# ============================================================
# Select best model: Random Forest With SES
# ============================================================

best_rf_ses = rf_outputs["With SES"]["model"]

print(best_rf_ses)
print(best_rf_ses.named_steps.keys())


In [ ]:
# ============================================================
# SHAP Analysis - Random Forest With SES
# ============================================================

shap.initjs()

# Best model from loaded evaluation outputs
best_rf_ses = rf_outputs["With SES"]["model"]

print("Pipeline steps:")
print(best_rf_ses.named_steps.keys())

preprocessor = best_rf_ses.named_steps["preprocess"]
rf_model = best_rf_ses.named_steps["model"]


In [ ]:
# ============================================================
# Get transformed feature names
# ============================================================

try:
    feature_names = list(preprocessor.get_feature_names_out())
except Exception:
    feature_names = []

    # Target-encoded high-cardinality feature(s)
    feature_names.extend(high_cardinality_features_ses)

    # One-hot encoded low-cardinality features
    onehot_encoder = preprocessor.named_transformers_["onehot_cat"]
    onehot_features = onehot_encoder.get_feature_names_out(low_cardinality_features_ses)
    feature_names.extend(list(onehot_features))

    # Numeric SES features
    feature_names.extend(numeric_features_ses)

print("Number of transformed features:", len(feature_names))
print(feature_names[:30])


In [ ]:
# ============================================================
# Transform data for SHAP
# ============================================================

X_train_transformed = preprocessor.transform(X_train_ses)
X_test_transformed = preprocessor.transform(X_test_ses)

if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

print("X_train_transformed:", X_train_transformed.shape)
print("X_test_transformed:", X_test_transformed.shape)
print("feature_names:", len(feature_names))

if X_test_transformed.shape[1] != len(feature_names):
    print("WARNING: feature_names length does not match transformed data columns.")


In [ ]:
# ============================================================
# Sample test data for SHAP
# ============================================================

shap_sample_size = 1000

rng = np.random.default_rng(42)
sample_idx = rng.choice(
    X_test_transformed.shape[0],
    size=min(shap_sample_size, X_test_transformed.shape[0]),
    replace=False
)

X_shap = X_test_transformed[sample_idx]
X_shap_df = pd.DataFrame(X_shap, columns=feature_names)

print("X_shap_df:", X_shap_df.shape)
display(X_shap_df.head())


In [ ]:
# ============================================================
# Compute SHAP values
# ============================================================

explainer = shap.TreeExplainer(rf_model)

shap_values_raw = explainer.shap_values(X_shap_df)

# Binary classification handling
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw[1]
elif len(shap_values_raw.shape) == 3:
    shap_values = shap_values_raw[:, :, 1]
else:
    shap_values = shap_values_raw

print("SHAP values shape:", shap_values.shape)


In [ ]:
# ============================================================
# SHAP mean absolute importance table
# ============================================================

shap_importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance_df.head(30))

shap_importance_df.to_csv(
    f"{save_dir}/shap_importance_random_forest_with_ses.csv",
    index=False
)


In [ ]:
# ============================================================
# SHAP summary bar plot
# ============================================================

plt.figure()
shap.summary_plot(
    shap_values,
    X_shap_df,
    plot_type="bar",
    max_display=20,
    show=False
)

plt.title("SHAP Feature Importance - Random Forest With SES")
plt.tight_layout()

plt.savefig(
    f"{save_dir}/shap_summary_bar_random_forest_with_ses.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# SHAP beeswarm plot
# ============================================================

plt.figure()
shap.summary_plot(
    shap_values,
    X_shap_df,
    max_display=20,
    show=False
)

plt.title("SHAP Summary Plot - Random Forest With SES")
plt.tight_layout()

plt.savefig(
    f"{save_dir}/shap_beeswarm_random_forest_with_ses.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# SES feature SHAP importance
# ============================================================

ses_keywords = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

ses_shap_importance = shap_importance_df[
    shap_importance_df["feature"].apply(
        lambda x: any(keyword in x for keyword in ses_keywords)
    )
].copy()

display(ses_shap_importance)

ses_shap_importance.to_csv(
    f"{save_dir}/shap_importance_ses_features_random_forest.csv",
    index=False
)


In [ ]:
# ============================================================
# SHAP dependence plots for SES variables
# ============================================================

for keyword in ["poverty_rate", "median_household_income"]:
    matching_features = [f for f in feature_names if keyword in f]

    if len(matching_features) == 0:
        print(f"No matching feature found for: {keyword}")
        continue

    feature = matching_features[0]
    print("Plotting:", feature)

    plt.figure()
    shap.dependence_plot(
        feature,
        shap_values,
        X_shap_df,
        show=False
    )

    plt.title(f"SHAP Dependence Plot: {keyword}")
    plt.tight_layout()

    plt.savefig(
        f"{save_dir}/shap_dependence_{keyword}_random_forest_with_ses.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


In [ ]:
# ============================================================
# Check saved SHAP files
# ============================================================

for f in sorted(os.listdir(save_dir)):
    if "shap" in f.lower():
        print(f)


In [ ]:
with open(f"{save_dir}/model_splits_preprocessed.pkl", "rb") as f:
    split_data = pickle.load(f)

best_rf_ses = rf_outputs["With SES"]["model"]
X_test_ses


In [ ]:
print("shap_values shape:", shap_values.shape)

print("min:", np.min(shap_values))
print("max:", np.max(shap_values))
print("mean abs:", np.mean(np.abs(shap_values)))
print("median abs:", np.median(np.abs(shap_values)))
print("95th percentile abs:", np.percentile(np.abs(shap_values), 95))
print("99th percentile abs:", np.percentile(np.abs(shap_values), 99))


In [ ]:
# ============================================================
# Check SHAP range by feature
# ============================================================

shap_range_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    "min_shap": shap_values.min(axis=0),
    "max_shap": shap_values.max(axis=0),
    "p95_abs_shap": np.percentile(np.abs(shap_values), 95, axis=0),
    "p99_abs_shap": np.percentile(np.abs(shap_values), 99, axis=0)
}).sort_values("mean_abs_shap", ascending=False)

display(shap_range_df.head(30))


In [ ]:
# ============================================================
# SHAP bar plot with larger figure
# ============================================================

plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    X_shap_df,
    plot_type="bar",
    max_display=20,
    show=False
)

plt.title("SHAP Feature Importance - Random Forest With SES")
plt.tight_layout()

plt.savefig(
    f"{save_dir}/shap_summary_bar_random_forest_with_ses_large.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# Clean SHAP feature names for thesis figure
# ============================================================

def clean_feature_name(name):
    name = name.replace("target_breed__", "")
    name = name.replace("onehot_cat__", "")
    name = name.replace("num__", "")

    name = name.replace("Animal_Breed_clean", "Animal breed")
    name = name.replace("Intake_Year_", "Intake year: ")
    name = name.replace("Animal_Origin_fe_", "Animal origin: ")
    name = name.replace("Intake_Type_", "Intake type: ")
    name = name.replace("Intake_Condition_fe_", "Intake condition: ")
    name = name.replace("Intake_Condition_clean_", "Intake condition: ")
    name = name.replace("Intake_Subtype_fe_", "Intake subtype: ")
    name = name.replace("Chip_Status_fe_", "Chip status: ")
    name = name.replace("Animal_Type_", "Animal type: ")
    name = name.replace("Animal_Size_", "Animal size: ")
    name = name.replace("Intake_Month_", "Intake month: ")

    name = name.replace("median_household_income", "Median household income")
    name = name.replace("bachelor_or_higher_pct", "Bachelor's degree or higher (%)")
    name = name.replace("poverty_rate", "Poverty rate")
    name = name.replace("unemployment_rate", "Unemployment rate")

    name = name.replace("_", " ")
    return name


shap_importance_plot_df = shap_importance_df.copy()
shap_importance_plot_df["feature_clean"] = shap_importance_plot_df["feature"].apply(clean_feature_name)

display(shap_importance_plot_df.head(20))


In [ ]:
# ============================================================
# Thesis-ready SHAP bar plot
# ============================================================

top_n = 15

plot_df = (
    shap_importance_plot_df
    .head(top_n)
    .sort_values("mean_abs_shap", ascending=True)
)

plt.figure(figsize=(9, 7))

plt.barh(
    plot_df["feature_clean"],
    plot_df["mean_abs_shap"]
)

plt.xlabel("Mean absolute SHAP value")
plt.ylabel("")
plt.title("SHAP Feature Importance: Random Forest with SES")

plt.tight_layout()

plt.savefig(
    f"{save_dir}/shap_feature_importance_rf_with_ses_clean.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# Aggregate SHAP importance to original feature level
# ============================================================

def map_to_original_feature(name):
    # Target encoded high-cardinality feature
    if "Animal_Breed_clean" in name:
        return "Animal breed"

    # Numeric SES features
    if "median_household_income" in name:
        return "Median household income"
    if "bachelor_or_higher_pct" in name:
        return "Bachelor's degree or higher (%)"
    if "poverty_rate" in name:
        return "Poverty rate"
    if "unemployment_rate" in name:
        return "Unemployment rate"

    # One-hot encoded original categorical variables
    if "Intake_Year" in name:
        return "Intake year"
    if "Intake_Month" in name:
        return "Intake month"
    if "Animal_Origin_fe" in name:
        return "Animal origin"
    if "Intake_Type" in name:
        return "Intake type"
    if "Intake_Subtype_fe" in name:
        return "Intake subtype"
    if "Intake_Condition_fe" in name or "Intake_Condition_clean" in name:
        return "Intake condition"
    if "Chip_Status_fe" in name:
        return "Chip status"
    if "Animal_Type" in name:
        return "Animal type"
    if "Animal_Size" in name:
        return "Animal size"

    return name


shap_original_df = shap_importance_df.copy()
shap_original_df["original_feature"] = shap_original_df["feature"].apply(map_to_original_feature)

shap_original_importance = (
    shap_original_df
    .groupby("original_feature", as_index=False)["mean_abs_shap"]
    .sum()
    .sort_values("mean_abs_shap", ascending=False)
)

display(shap_original_importance)


In [ ]:
# ============================================================
# Thesis-ready aggregated SHAP plot
# ============================================================

plot_df = shap_original_importance.sort_values("mean_abs_shap", ascending=True)

plt.figure(figsize=(8, 6))

plt.barh(
    plot_df["original_feature"],
    plot_df["mean_abs_shap"]
)

plt.xlabel("Aggregated mean absolute SHAP value")
plt.ylabel("")
plt.title("Aggregated SHAP Feature Importance: Random Forest with SES")

plt.tight_layout()

plt.savefig(
    f"{save_dir}/shap_feature_importance_rf_with_ses_aggregated.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# SES-only SHAP importance plot
# ============================================================

ses_features_clean = [
    "Median household income",
    "Bachelor's degree or higher (%)",
    "Poverty rate",
    "Unemployment rate"
]

ses_plot_df = shap_original_importance[
    shap_original_importance["original_feature"].isin(ses_features_clean)
].sort_values("mean_abs_shap", ascending=True)

display(ses_plot_df)

plt.figure(figsize=(7, 4))

plt.barh(
    ses_plot_df["original_feature"],
    ses_plot_df["mean_abs_shap"]
)

plt.xlabel("Aggregated mean absolute SHAP value")
plt.ylabel("")
plt.title("SHAP Importance of SES Features")

plt.tight_layout()

plt.savefig(
    f"{save_dir}/shap_importance_ses_features_rf.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### 10.1 Re-plot Saved SHAP Figures


#### Load Saved SHAP Data


In [ ]:
# Load saved SHAP outputs - Random Forest With SES

save_dir = "/content/drive/MyDrive/Thesis"
shap_dir = f"{save_dir}/shap_rf_with_ses"

print("SHAP directory exists:", os.path.exists(shap_dir))
print("Files in SHAP directory:")
for f in sorted(os.listdir(shap_dir)):
    print(f)


In [ ]:
# ============================================================
# Load SHAP values and sample data
# ============================================================

shap_values = np.load(
    f"{shap_dir}/shap_values_rf_with_ses.npy"
)

X_shap_df = pd.read_csv(
    f"{shap_dir}/X_shap_sample_rf_with_ses.csv"
)

shap_values_df = pd.read_csv(
    f"{shap_dir}/shap_values_rf_with_ses.csv"
)

print("shap_values shape:", shap_values.shape)
print("X_shap_df shape:", X_shap_df.shape)
print("shap_values_df shape:", shap_values_df.shape)

display(X_shap_df.head())
display(shap_values_df.head())


In [ ]:
# ============================================================
# Load SHAP importance tables
# ============================================================

shap_importance_df = pd.read_csv(
    f"{shap_dir}/shap_importance_rf_with_ses_transformed_features.csv"
)

display(shap_importance_df.head(20))


In [ ]:
# ============================================================
# Load optional SHAP outputs if they exist
# ============================================================

def load_csv_if_exists(path):
    if os.path.exists(path):
        print("Loaded:", os.path.basename(path))
        return pd.read_csv(path)
    else:
        print("Missing:", os.path.basename(path))
        return None


shap_importance_plot_df = load_csv_if_exists(
    f"{shap_dir}/shap_importance_rf_with_ses_clean_feature_names.csv"
)

shap_original_importance = load_csv_if_exists(
    f"{shap_dir}/shap_importance_rf_with_ses_aggregated_original_features.csv"
)

ses_shap_importance = load_csv_if_exists(
    f"{shap_dir}/shap_importance_rf_with_ses_ses_features.csv"
)

ses_shap_range = load_csv_if_exists(
    f"{shap_dir}/shap_range_rf_with_ses_ses_features.csv"
)

shap_range_df = load_csv_if_exists(
    f"{shap_dir}/shap_range_rf_with_ses_transformed_features.csv"
)


In [ ]:
# ============================================================
# Load SHAP metadata
# ============================================================

with open(f"{shap_dir}/shap_metadata_rf_with_ses.json", "r") as f:
    shap_metadata = json.load(f)

shap_metadata


#### SHAP Summary Plot


In [ ]:
# Re-plot SHAP summary plot from loaded outputs

figure_dir = f"{save_dir}/figures"
os.makedirs(figure_dir, exist_ok=True)

plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    X_shap_df,
    max_display=15,
    show=False,
    plot_size=None
)

plt.title("SHAP Summary Plot: Random Forest with SES", fontsize=13, pad=12)
plt.xlabel("SHAP value impact on long-stay prediction", fontsize=11)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()

plt.savefig(
    f"{figure_dir}/rf-shap-summary-loaded.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# Re-plot aggregated SHAP feature importance
# ============================================================

if shap_original_importance is not None:
    plot_df = shap_original_importance.sort_values("mean_abs_shap", ascending=True)

    plt.figure(figsize=(8.5, 5.8))

    plt.barh(
        plot_df["original_feature"],
        plot_df["mean_abs_shap"],
        color="white",
        edgecolor="black",
        hatch="///",
        linewidth=0.8
    )

    plt.xlabel("Aggregated mean absolute SHAP value", fontsize=12)
    plt.ylabel("")
    plt.title("Aggregated SHAP Feature Importance", fontsize=13, pad=12)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)

    plt.tight_layout()

    plt.savefig(
        f"{figure_dir}/rf-shap-aggregated.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
else:
    print("shap_original_importance was not saved or not loaded.")


In [ ]:
# Save both SHAP plots for thesis
# 1) SHAP summary / beeswarm plot
# 2) Aggregated SHAP feature importance plot


figure_dir = f"{save_dir}/figures"
os.makedirs(figure_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. Clean feature names for SHAP summary plot
# ------------------------------------------------------------

def clean_shap_feature_name(name):
    name = str(name)

    name = name.replace("target_breed__", "")
    name = name.replace("onehot_cat__", "")
    name = name.replace("num__", "")

    name = name.replace("Animal_Breed_clean", "Animal breed")
    name = name.replace("Intake_Year_", "Intake year: ")
    name = name.replace("Intake_Month_", "Intake month: ")
    name = name.replace("Animal_Origin_fe_", "Animal origin: ")
    name = name.replace("Intake_Type_", "Intake type: ")
    name = name.replace("Intake_Subtype_fe_", "Intake subtype: ")
    name = name.replace("Intake_Condition_clean_", "Intake condition: ")
    name = name.replace("Intake_Condition_fe_", "Intake condition: ")
    name = name.replace("Chip_Status_fe_", "Chip status: ")
    name = name.replace("Animal_Type_", "Animal type: ")
    name = name.replace("Animal_Size_", "Animal size: ")

    name = name.replace("median_household_income", "Median household income")
    name = name.replace("bachelor_or_higher_pct", "Bachelor's degree or higher (%)")
    name = name.replace("poverty_rate", "Poverty rate")
    name = name.replace("unemployment_rate", "Unemployment rate")

    name = name.replace("_", " ")

    return name


X_shap_plot = X_shap_df.copy()
X_shap_plot.columns = [clean_shap_feature_name(c) for c in X_shap_plot.columns]


# ------------------------------------------------------------
# 2. Save SHAP summary / beeswarm plot
# ------------------------------------------------------------

plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    X_shap_plot,
    max_display=15,
    show=False,
    plot_size=None
)

plt.title("SHAP Summary Plot: Random Forest with SES", fontsize=13, pad=12)
plt.xlabel("SHAP value impact on long-stay prediction", fontsize=11)

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()

plt.savefig(
    f"{figure_dir}/rf-shap-summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{figure_dir}/rf-shap-summary.pdf",
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 3. Save aggregated SHAP feature importance plot
# ------------------------------------------------------------

if "shap_original_importance" not in globals():
    raise ValueError("shap_original_importance does not exist. Run the aggregated SHAP importance code first.")

plot_df = shap_original_importance.sort_values(
    "mean_abs_shap",
    ascending=True
)

plt.figure(figsize=(8.5, 5.8))

plt.barh(
    plot_df["original_feature"],
    plot_df["mean_abs_shap"],
    color="white",
    edgecolor="black",
    hatch="///",
    linewidth=0.8
)

plt.xlabel("Aggregated mean absolute SHAP value", fontsize=12)
plt.ylabel("")
plt.title("Aggregated SHAP Feature Importance", fontsize=13, pad=12)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)

plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()

plt.savefig(
    f"{figure_dir}/rf-shap-aggregated.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{figure_dir}/rf-shap-aggregated.pdf",
    bbox_inches="tight"
)

plt.show()


print("Saved SHAP plots:")
print(f"{figure_dir}/rf-shap-summary.png")
print(f"{figure_dir}/rf-shap-summary.pdf")
print(f"{figure_dir}/rf-shap-aggregated.png")
print(f"{figure_dir}/rf-shap-aggregated.pdf")


#### SHAP Dependence Plot


In [ ]:
# ============================================================
# SHAP dependence plot - Median Household Income
# Random Forest With Socioeconomic Variables
# ============================================================

import os
import matplotlib.pyplot as plt

# Save directory
fig_dir = f"{save_dir}/figures"
os.makedirs(fig_dir, exist_ok=True)

# ------------------------------------------------------------
# Check available columns
# ------------------------------------------------------------

print("X_shap_df columns:")
print(X_shap_df.columns.tolist())

print("\nshap_values_df columns:")
print(shap_values_df.columns.tolist())

# ------------------------------------------------------------
# Find median household income column
# ------------------------------------------------------------

income_candidates = [
    "median_household_income",
    "Median household income",
    "median household income",
    "median_household_income_scaled",
]

income_x_col = None
income_shap_col = None

for col in X_shap_df.columns:
    if "median" in col.lower() and "income" in col.lower():
        income_x_col = col
        break

for col in shap_values_df.columns:
    if "median" in col.lower() and "income" in col.lower():
        income_shap_col = col
        break

print("Selected X column:", income_x_col)
print("Selected SHAP column:", income_shap_col)

if income_x_col is None:
    raise ValueError("Could not find median household income column in X_shap_df.")

if income_shap_col is None:
    raise ValueError("Could not find median household income SHAP column in shap_values_df.")

# ------------------------------------------------------------
# Create dependence plot
# ------------------------------------------------------------

plot_df = X_shap_df[[income_x_col]].copy()
plot_df["shap_value"] = shap_values_df[income_shap_col]

plt.figure(figsize=(7, 5))

plt.scatter(
    plot_df[income_x_col],
    plot_df["shap_value"],
    alpha=0.35,
    s=18
)

plt.axhline(0, linewidth=1)
plt.xlabel("Median household income")
plt.ylabel("SHAP value impact on long-stay prediction")
plt.title("SHAP Dependence Plot: Median Household Income")

plt.tight_layout()

# Save
plt.savefig(
    f"{fig_dir}/shap-dependence-median-household-income.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    f"{fig_dir}/shap-dependence-median-household-income.pdf",
    bbox_inches="tight"
)

plt.show()


## 11. Save Thesis Tables and Figures


### 11.1 Save SHAP Outputs


In [ ]:
# ============================================================
# Save all SHAP outputs - Random Forest With SES
# ============================================================


shap_dir = f"{save_dir}/shap_rf_with_ses"
os.makedirs(shap_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. Save SHAP values and SHAP sample data
# ------------------------------------------------------------

np.save(
    f"{shap_dir}/shap_values_rf_with_ses.npy",
    shap_values
)

X_shap_df.to_csv(
    f"{shap_dir}/X_shap_sample_rf_with_ses.csv",
    index=False
)

# ------------------------------------------------------------
# 2. Save raw SHAP values as CSV
#    Rows = sampled test observations
#    Columns = transformed features
# ------------------------------------------------------------

shap_values_df = pd.DataFrame(
    shap_values,
    columns=X_shap_df.columns
)

shap_values_df.to_csv(
    f"{shap_dir}/shap_values_rf_with_ses.csv",
    index=False
)

# ------------------------------------------------------------
# 3. Save SHAP feature importance table
# ------------------------------------------------------------

shap_importance_df.to_csv(
    f"{shap_dir}/shap_importance_rf_with_ses_transformed_features.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Save cleaned feature-name table if available
# ------------------------------------------------------------

if "shap_importance_plot_df" in globals():
    shap_importance_plot_df.to_csv(
        f"{shap_dir}/shap_importance_rf_with_ses_clean_feature_names.csv",
        index=False
    )

# ------------------------------------------------------------
# 5. Save aggregated original-feature importance if available
# ------------------------------------------------------------

if "shap_original_importance" in globals():
    shap_original_importance.to_csv(
        f"{shap_dir}/shap_importance_rf_with_ses_aggregated_original_features.csv",
        index=False
    )

# ------------------------------------------------------------
# 6. Save SES-only SHAP importance if available
# ------------------------------------------------------------

if "ses_shap_importance" in globals():
    ses_shap_importance.to_csv(
        f"{shap_dir}/shap_importance_rf_with_ses_ses_features.csv",
        index=False
    )

if "ses_shap_range" in globals():
    ses_shap_range.to_csv(
        f"{shap_dir}/shap_range_rf_with_ses_ses_features.csv",
        index=False
    )

# ------------------------------------------------------------
# 7. Save SHAP range table if available
# ------------------------------------------------------------

if "shap_range_df" in globals():
    shap_range_df.to_csv(
        f"{shap_dir}/shap_range_rf_with_ses_transformed_features.csv",
        index=False
    )

# ------------------------------------------------------------
# 8. Save metadata
# ------------------------------------------------------------

shap_metadata = {
    "model": "Random Forest Tuned",
    "feature_set": "With SES",
    "shap_sample_size": int(X_shap_df.shape[0]),
    "n_transformed_features": int(X_shap_df.shape[1]),
    "shap_values_shape": list(shap_values.shape),
    "shap_value_min": float(np.min(shap_values)),
    "shap_value_max": float(np.max(shap_values)),
    "mean_abs_shap": float(np.mean(np.abs(shap_values))),
    "median_abs_shap": float(np.median(np.abs(shap_values))),
    "p95_abs_shap": float(np.percentile(np.abs(shap_values), 95)),
    "p99_abs_shap": float(np.percentile(np.abs(shap_values), 99)),
    "note": "SHAP values were computed for the positive class, corresponding to long-stay risk."
}

with open(f"{shap_dir}/shap_metadata_rf_with_ses.json", "w") as f:
    json.dump(shap_metadata, f, indent=4)

print("Saved SHAP tables and arrays to:")
print(shap_dir)
